# 03. Autograd Mechanics, Complex Wirtinger & Custom Functions

- **Category:** 00. Foundations & Automatic Differentiation
- **Lineage & Citations:** Wirtinger (1927) *Zur formalen Theorie der Funktionen von mehr komplexen Veränderlichen*; Linnainmaa (1970) *Taylor expansion of the accumulated rounding error*; Rumelhart, Hinton & Williams (1986) *Learning representations by back-propagating errors*.
- **Purpose:** Implements reverse-mode automatic differentiation internals: custom `autograd.Function` with `setup_context`, analytical Vector-Jacobian Products (VJPs), and complex Wirtinger calculus for non-holomorphic real-valued loss objectives.

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> 1. Do not mutate tensors in-place after they have been saved in `ctx.save_for_backward()`; doing so corrupts gradient evaluation and triggers a `RuntimeError`.
> 2. Do not accumulate loss tensors across training loops via `total += loss`; this retains the entire computational graph in VRAM, causing OOM crashes. Always use `total += loss.item()`.
> 3. Do not apply standard complex derivatives to real-valued losses; real losses are non-holomorphic and require Wirtinger conjugate derivatives.

## Mathematical Formulation & Complexity

### 1. Vector-Jacobian Product (VJP)
Given $f: \mathbb{R}^N \to \mathbb{R}^M$ and loss $\mathcal{L}: \mathbb{R}^M \to \mathbb{R}$ with upstream gradient $v = \nabla_f \mathcal{L}$:
$$\nabla_x \mathcal{L} = v^\top J, \quad J_{ij} = \frac{\partial f_i}{\partial x_j}$$
For $Y = X W + b$: $dX = G W^\top, \ dW = X^\top G, \ db = \sum G$.

### 2. Complex Wirtinger Calculus
For $z = x + i y \in \mathbb{C}$, the Wirtinger differential operators treat $z$ and $z^*$ as independent:
$$\frac{\partial}{\partial z} = \frac{1}{2}\left(\frac{\partial}{\partial x} - i\frac{\partial}{\partial y}\right), \quad \frac{\partial}{\partial z^*} = \frac{1}{2}\left(\frac{\partial}{\partial x} + i\frac{\partial}{\partial y}\right)$$

PyTorch autograd defines `z.grad` as $2 \frac{\partial \mathcal{L}}{\partial z^*} = \frac{\partial \mathcal{L}}{\partial x} + i \frac{\partial \mathcal{L}}{\partial y}$ so that complex descent $z \leftarrow z - \eta \cdot \text{z.grad}$ updates real and imaginary components by their respective real gradients.

**Asymptotic Complexity:** Reverse-mode VJP Time: $\mathcal{O}(\text{OPS}(f))$; Tape Memory: $\mathcal{O}(\text{Activations Saved})$.

In [ ]:
import torch

import core

# 1. Deterministic Environment Setup
env = core.setup_env(seed=42)

# 2. Real and Complex Synthetic Test Tensors
b_dim, in_dim, out_dim = 4, 3, 2
x_real = torch.randn(b_dim, in_dim, dtype=torch.float64, requires_grad=True)
w_real = torch.randn(in_dim, out_dim, dtype=torch.float64, requires_grad=True)
b_real = torch.randn(out_dim, dtype=torch.float64, requires_grad=True)

x_comp = torch.complex(torch.randn(b_dim, in_dim), torch.randn(b_dim, in_dim))
w_comp = torch.complex(torch.randn(in_dim, out_dim), torch.randn(in_dim, out_dim)).requires_grad_(
    True
)
y_comp = torch.complex(torch.randn(b_dim, out_dim), torch.randn(b_dim, out_dim))

print(f"Device: {env.device} | Real: {x_real.shape} | Complex: {x_comp.shape}")

In [ ]:
# 1. Custom Autograd Function with setup_context
class ScaledLinearFunction(torch.autograd.Function):
    @staticmethod
    def forward(input_tensor, weight, bias, scale=1.0):
        return scale * (input_tensor @ weight + bias)

    @staticmethod
    def setup_context(ctx, inputs, output):
        input_tensor, weight, bias, scale = inputs
        ctx.save_for_backward(input_tensor, weight)
        ctx.has_bias = bias is not None
        ctx.scale = scale

    @staticmethod
    def backward(ctx, grad_output):
        input_tensor, weight = ctx.saved_tensors
        scale = ctx.scale
        scaled_g = grad_output * scale
        grad_input = scaled_g @ weight.T if ctx.needs_input_grad[0] else None
        grad_weight = input_tensor.T @ scaled_g if ctx.needs_input_grad[1] else None
        grad_bias = scaled_g.sum(dim=0) if (ctx.needs_input_grad[2] and ctx.has_bias) else None
        return grad_input, grad_weight, grad_bias, None


def scaled_linear(x, w, b, scale=1.0):
    return ScaledLinearFunction.apply(x, w, b, scale)


pred_real = scaled_linear(x_real, w_real, b_real, scale=1.5)
print(f"Custom Autograd Output Shape: {pred_real.shape}")

In [ ]:
# Quantitative Verification & Mathematical Invariant Checks
# 1. Gradcheck Validation vs Finite Differences
test_passed = torch.autograd.gradcheck(
    scaled_linear,
    (x_real, w_real, b_real, 1.5),
    eps=1e-6,
    atol=1e-4,
    rtol=1e-4,
)
assert test_passed, "Gradcheck failed!"
print("Gradcheck passed: Custom analytical VJP matches finite difference approximations.")

# 2. Complex Wirtinger Calculus Invariant
z = x_comp @ w_comp
diff = z - y_comp
loss = 0.5 * torch.sum(diff.real**2 + diff.imag**2)
loss.backward()

# Analytical Wirtinger gradient: PyTorch stores 2 * dL/dz* = (z - y)
# dL/dW* = X^H @ (z - y)
grad_w_analytic = x_comp.mH @ (z - y_comp)
torch.testing.assert_close(w_comp.grad, grad_w_analytic, atol=1e-5, rtol=1e-5)
print("Complex Wirtinger gradient verified: w.grad matches X^H @ (z - y) exactly.")

In [ ]:
# Standardized Performance Benchmark Table
from core.benchmark import benchmark_fn, format_benchmark_table

nn_linear = torch.nn.Linear(in_dim, out_dim, dtype=torch.float64)
g_real = torch.ones_like(pred_real)


def run_custom():
    out = scaled_linear(x_real, w_real, b_real, 1.0)
    out.backward(g_real, retain_graph=True)


def run_native():
    out = nn_linear(x_real)
    out.backward(g_real, retain_graph=True)


b_custom = benchmark_fn(run_custom, iters=30, name="Custom setup_context VJP")
b_native = benchmark_fn(run_native, iters=30, name="torch.nn.Linear Autograd")

print(format_benchmark_table([b_custom, b_native]))

In [ ]:
# Runnable Misuse Contrast: In-Place Mutation of Saved Tensors
# FAILURE MODE: Mutating a tensor saved for backward triggers a fatal runtime check!
class InplaceDangerousFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x * 2.0

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2.0


x_danger = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y_danger = InplaceDangerousFunction.apply(x_danger)

# Corrupt x_danger in-place before backward!
x_danger.data.add_(10.0)
try:
    y_danger.backward(torch.ones_like(y_danger))
    print("Warning: In-place modification was not caught directly!")
except RuntimeError as e:
    print(f"[FAILURE CAUGHT] Autograd successfully detected in-place mutation: {e}")

# CORRECT PATTERN: Use out-of-place or clone
x_safe = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y_safe = InplaceDangerousFunction.apply(x_safe.clone())
y_safe.backward(torch.ones_like(y_safe))
print(f"[CORRECT] Safe backward gradient computed: {x_safe.grad.tolist()}")

## Architectural Tape, Key Takeaways & Relative Navigation

### Autograd Execution Tape Topology
<svg width="650" height="150" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <defs>
    <marker id="arrow" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#38bdf8"/>
    </marker>
    <marker id="arrow-back" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#f43f5e"/>
    </marker>
  </defs>
  <text x="30" y="25" fill="#94a3b8" font-size="12" font-weight="bold">AUTOGRAD TAPE &amp; VECTOR-JACOBIAN PRODUCT (VJP)</text>
  <!-- Forward Graph Nodes -->
  <rect x="30" y="40" width="80" height="30" rx="4" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="70" y="60" fill="#f8fafc" font-size="12" text-anchor="middle">Input X</text>
  <line x1="110" y1="55" x2="160" y2="55" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow)"/>
  <rect x="160" y="40" width="120" height="30" rx="4" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="220" y="60" fill="#f8fafc" font-size="11" text-anchor="middle">Forward: X · W + b</text>
  <line x1="280" y1="55" x2="330" y2="55" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow)"/>
  <rect x="330" y="40" width="80" height="30" rx="4" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="370" y="60" fill="#f8fafc" font-size="12" text-anchor="middle">Loss L</text>
  <!-- Backward Graph Nodes -->
  <line x1="370" y1="70" x2="370" y2="95" stroke="#f43f5e" stroke-width="2" marker-end="url(#arrow-back)"/>
  <rect x="300" y="95" width="140" height="30" rx="4" fill="#1e293b" stroke="#f43f5e" stroke-width="1.5"/>
  <text x="370" y="115" fill="#f43f5e" font-size="11" text-anchor="middle">Upstream Adjoint G</text>
  <line x1="300" y1="110" x2="230" y2="110" stroke="#f43f5e" stroke-width="2" marker-end="url(#arrow-back)"/>
  <rect x="110" y="95" width="120" height="30" rx="4" fill="#1e293b" stroke="#f43f5e" stroke-width="1.5"/>
  <text x="170" y="115" fill="#f43f5e" font-size="11" text-anchor="middle">VJP: G · Wᵀ</text>
  <text x="460" y="60" fill="#38bdf8" font-size="11">Forward Pass (Saves X, W)</text>
  <text x="460" y="115" fill="#f43f5e" font-size="11">Backward Pass (Evaluates VJP)</text>
</svg>

### Key Takeaways
1. **Reverse-Mode Efficiency:** Computing $\nabla_x \mathcal{L}$ for a scalar loss takes identical computational complexity as the forward pass, regardless of input dimension $N$.
2. **Memory Conservation:** Use PyTorch 2.x `setup_context` to save only essential tensors in `ctx.save_for_backward()`, avoiding retaining large intermediate tensors.
3. **Complex Gradient Convention:** PyTorch autograd evaluates $2 \frac{\partial \mathcal{L}}{\partial z^*}$, ensuring complex parameters update in the direction of steepest descent.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_autograd_mechanics_complex_wirtinger_and_custom_functions.md](../../markdown/00_foundations/03_autograd_mechanics_complex_wirtinger_and_custom_functions.md)
- **PyTorch 2.x Production Module:** [03_autograd_mechanics_complex_wirtinger_and_custom_functions.py](../../pytorch2/00_foundations/03_autograd_mechanics_complex_wirtinger_and_custom_functions.py)
- **Functional JAX Track:** [03_autograd_mechanics_complex_wirtinger_and_custom_functions.py](../../jax/00_foundations/03_autograd_mechanics_complex_wirtinger_and_custom_functions.py)
- **Native Rust Candle Track:** [03_autograd_mechanics_complex_wirtinger_and_custom_functions.rs](../../candle/00_foundations/03_autograd_mechanics_complex_wirtinger_and_custom_functions.rs)
- **CPU Fused Kernel:** [03_autograd_mechanics_complex_wirtinger_and_custom_functions_cpu.py](../../kernels/00_foundations/03_autograd_mechanics_complex_wirtinger_and_custom_functions_cpu.py)
- **Golden Reference Checkpoint:** [03_autograd_mechanics_complex_wirtinger_and_custom_functions.safetensors](../../weights/00_foundations/03_autograd_mechanics_complex_wirtinger_and_custom_functions.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. Tensors, Views, Strides & Memory Layouts](02_tensors_views_strides_and_memory_layouts.ipynb) | [Master Index: README.md](../../README.md) | [Next: 04. Forward AD & Functional Transforms](04_forward_ad_and_functional_transforms_torch_func.ipynb) &rarr;
